# Module 1.3: Post-Boot Security Hardening & Password Rotation

## Homelab NAS Development - Step 1: Operating System

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand Linux authentication architecture, Pluggable Authentication Modules (PAM), and the shadow password database (`/etc/shadow`).
2. Explain the security threat model of bootstrap imaging credentials on shared local networks.
3. Execute the `passwd` utility safely and understand silent terminal input (echo suppression).
4. Invalidate cached administrative tokens using `sudo -k` and verify privilege elevation with `sudo whoami`.
5. Apply the cardinal rule of remote system administration: verify new credentials in an independent parallel SSH session before disconnecting.

## 1.3.1 Conceptual Background: Linux Authentication & Credential Hardening

When provisioning a headless server, the initial credentials supplied during image flashing serve exclusively as **bootstrapping secrets**. Leaving these initial credentials unchanged—or utilizing standard default passwords—exposes your NAS to unauthorized lateral movement from compromised devices on your local network.

### Linux PAM (Pluggable Authentication Modules)
User authentication in Debian and Raspberry Pi OS is handled modularly through **PAM**. When a login request occurs (whether via local console, SSH, or `sudo`), PAM dynamically chains shared library modules defined in `/etc/pam.d/`:
- `pam_unix.so`: Manages traditional UNIX password authentication against `/etc/shadow`.
- `pam_faildelay.so`: Imposes artificial delays on failed login attempts to thwart automated brute-force attacks.

### Modern Password Hashing: Yescrypt in Debian 12 (Bookworm)
Passphrases are never stored in plaintext. In modern 64-bit Raspberry Pi OS (Debian 12 Bookworm), password hashes are stored in `/etc/shadow` using the modern **yescrypt** key derivation function (prefixed with `$y$`). 
- Yescrypt is designed to be **memory-hard**, requiring dedicated RAM for hash verification.
- This architectural choice renders offline dictionary attacks using GPUs, FPGAs, or ASIC rigs computationally intractable compared to legacy MD5 or single-round SHA-512 hashes.

### Sudo Privilege Architecture & Session Caching
The standard administrative user on Raspberry Pi OS belongs to the `sudo` group defined in `/etc/group` and `/etc/sudoers`.
- When an administrative command is executed with `sudo`, PAM validates the user's password and issues a temporary authorization timestamp token (stored in `/run/sudo/ts/`).
- By default, this token remains valid for **15 minutes** (`timestamp_timeout`), allowing sequential administrative commands without re-prompting.
- To test that a newly configured password functions for privilege elevation immediately, you must run `sudo -k` (kill cache), which invalidates the active timestamp and forces explicit re-authentication.

### The Golden Rule of Remote Administration
> ⚠️ **Cardinal Rule**: Whenever you modify user passwords, authentication credentials, or SSH daemon configurations on a remote headless host, **never close your current active terminal session** until you have successfully authenticated in an independent, secondary terminal window. If a typo occurred during the credential update, your active session remains logged in with root/sudo authority, allowing you to rectify the issue without having to connect physical monitors, keyboards, or reflash the OS.

## 📖 Official Hardware & Setup Reference Figures

<br>

<img src="../../assets/reference/linux_password_security.jpg" width="85%">

*Reference Figure 1: Linux PAM Architecture, /etc/shadow Yescrypt Hashing, Sudo Cache Invalidation, and Parallel SSH Verification Session*

<br>

## 1.3.2 Step-by-Step Laboratory Guide

### Task 1.3.1: Log into the Raspberry Pi via SSH
1. From your workstation terminal (macOS Terminal, Linux Bash, or Windows PowerShell), connect to your running Raspberry Pi using your initial bootstrap credentials:
   ```bash
   ssh pi@homelab-pi.local
   ```
2. Accept the host key fingerprint if this is your first time connecting.
3. Enter the bootstrap password configured in Raspberry Pi Imager.

---

### Task 1.3.2: Rotate Your User Password with `passwd`
1. Run the standard Linux password update utility:
   ```bash
   passwd
   ```
2. When prompted, enter your **Current password**.
3. Enter your **New password**.
   - *Note on terminal behavior*: Linux intentionally disables terminal echo (`stty -echo`) when reading passwords. No asterisks (`*`), bullets, or characters will appear on screen. Type your password deliberately and press Enter.
4. Retype your **New password** to confirm.
5. You should receive the confirmation message:
   ```text
   passwd: password updated successfully
   ```

---

### Task 1.3.3: Invalidate Cached Sudo Credentials & Verify Elevation
1. Force the `sudo` subsystem to revoke any active timestamp token:
   ```bash
   sudo -k
   ```
2. Run an administrative command to trigger an authentication challenge:
   ```bash
   sudo whoami
   ```
3. When prompted:
   ```text
   [sudo] password for pi:
   ```
   Enter your **newly established password**.
4. Confirm that the terminal outputs:
   ```text
   root
   ```
   This confirms that your new credentials are valid for root privilege elevation.

---

### Task 1.3.4: Verify Independent SSH Authentication (Parallel Session)
1. **DO NOT CLOSE** your current terminal session.
2. Open a **new, separate terminal window** on your workstation.
3. Initiate an SSH connection:
   ```bash
   ssh pi@homelab-pi.local
   ```
4. Authenticate using your **new password**.
5. Once your secondary terminal session opens successfully at the `pi@homelab-pi:~$` prompt, you have verified authentication end-to-end. You may now close either terminal window.

## 📸 Verification & Screenshot Submissions

*To confirm milestone completion, capture proof of your progress and insert your screenshots in the designated placeholders below.*

### 🖼️ Milestone Submission 1: Password Rotation & Sudo Verification

**Verification Objective:** Capture a screenshot of your terminal showing the successful execution of `passwd` and the `sudo -k` followed by `sudo whoami` returning `root`.

---

![Password Rotation & Sudo Verification](assets/screenshots/01_3_password_change_verification.png)

*Figure 1: Verification evidence for Password Rotation & Sudo Verification. Place your image at `assets/screenshots/01_3_password_change_verification.png`.*

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Initial bootstrap imaging password successfully rotated using `passwd`.
- [ ] New password meets high-entropy standards and is recorded in your secure password vault.
- [ ] Administrative privilege elevation verified using `sudo -k` and `sudo whoami`.
- [ ] Parallel secondary SSH session successfully authenticated before closing the primary terminal.

Proceed to [Module 2.1: LAN Discovery & Static IP Configuration](../../modules/02_network_configuration/02_1_network_discovery_and_ip.ipynb) once this checkpoint is satisfied.